In [107]:
import pandas as pd
import os
import sys
from tqdm.auto import tqdm
from time_series import create_time_series
sys.path.insert(0, os.path.abspath("../fsrs-optimizer/src/fsrs_optimizer/"))
import numpy as np

tqdm.pandas()

In [108]:
USER_ID = 2
df = pd.read_parquet(
    "../anki-revlogs-10k/revlogs", filters=[("user_id", "=", USER_ID), ("rating", "in", [1, 2, 3, 4])] 
)
df

,card_id,day_offset,rating,state,duration,elapsed_days,elapsed_seconds,user_id
27383,822,0,1,0,32765,-1,-1,2
27390,823,0,1,0,27479,-1,-1,2
27396,824,0,1,0,38333,-1,-1,2
27402,825,0,1,0,26448,-1,-1,2
27407,826,0,1,0,22190,-1,-1,2
...,...,...,...,...,...,...,...,...
69169,2701,3634,1,2,4194,11,1005030,2
71400,2849,3634,2,0,4682,-1,-1,2
63178,2376,3634,3,2,5182,24,2089579,2
70694,2805,3634,3,2,5410,5,434345,2


# Group

Group the reviews by reviews that happen with a < 1d interval

Then count how many good reviews there were at the end

In [109]:
from collections import defaultdict

groups = df.groupby("card_id")

data = defaultdict(lambda: (0, 0))

for card_id, card in groups:
    #print(card_id, "=",)# card)
    pass_count = 0
    for review in card.iloc:
        elapsed_seconds, elapsed_days, rating = review[["elapsed_seconds", "elapsed_days", "rating"]]
        if elapsed_days == 1:
            data[pass_count] = (data[pass_count][0] + int(rating > 1), data[pass_count][1] + 1)
            pass_count = 0
        elif elapsed_days == 0:
            if rating > 1:
                pass_count += 1
            else: 
                pass_count = 0
        else:
            pass_count = 0

        #print(f"{elapsed_seconds=}, {elapsed_days=}, {rating=}, {pass_count=}")

for i, data in sorted(data.items(), key =lambda data: data[0]):
    if data[0] > 0:
        print(i, data[0] / data[1], data[1])

0 0.8674609084139985 9401
1 0.8767990788716177 1737
2 0.9348370927318296 399
3 0.9807692307692307 156
4 0.9767441860465116 43
5 1.0 10
7 1.0 1
